# scVI via Hugging Face — a comparison path, not a replacement

This is a **separate experiment** from `scvi_pretraining.ipynb`, run in its own notebook/runtime so it doesn't interfere with the from-scratch training job.

**What this notebook does:** loads `scvi-tools/heart-cell-atlas-scvi` — a model already pretrained on 485,000 cardiac cells from a different atlas — from the Hugging Face Hub, and adapts it to our DCM/ACM cells via `scvi.model.SCVI.load_query_data()`. Then runs the same honest classification pipeline on the resulting latent space.

**Why this is a different question, not a faster version of the same one.** The pretraining notebook asks: does pretraining scVI *on our own atlas* produce features that classify better than raw genes? This notebook asks something else: does a representation pretrained on *someone else's* cardiac cohort, sequenced with a different protocol and a different (overlapping but not identical) gene panel, transfer usefully to our classification task at all? Both are legitimate questions. Their AUC numbers should not be read as directly comparable — they test different hypotheses that happen to share a downstream classifier.

The honest motivation for including this at all: I wanted to learn how the Hugging Face Hub and scvi-hub work, and this was a real way to do that rather than a toy example.

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout[:400] if result.returncode == 0 else 'No GPU — change runtime type first')

In [ ]:
%pip install -q scvi-tools huggingface_hub scanpy matplotlib xgboost
print('Done')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import anndata as ad, numpy as np, scanpy as sc, scipy.sparse as sp, warnings
warnings.filterwarnings('ignore')

H5AD_PATH = '/content/drive/MyDrive/cardiomyocytes.h5ad'  # CHANGE IF NEEDED
DCM = 'dilated cardiomyopathy'
ACM = 'arrhythmogenic right ventricular cardiomyopathy'

print('Loading (DCM+ACM only, same memory-safe approach as the pretraining notebook)...')
adata = ad.read_h5ad(H5AD_PATH, backed='r')
mask = adata.obs['disease'].isin([DCM, ACM])
adata = adata[mask].to_memory()
print(f'Loaded: {adata.n_obs:,} cells x {adata.n_vars:,} genes')
print(adata.obs['disease'].value_counts().to_string())

In [ ]:
# Download the pretrained model from the Hugging Face Hub via scvi-hub.
# This is the actual Hugging Face usage: pulling someone else's trained
# weights + metadata instead of training from scratch.
from scvi.hub import HubModel

print('Downloading scvi-tools/heart-cell-atlas-scvi from Hugging Face...')
hub_model = HubModel.pull_from_huggingface_hub(
    repo_name='scvi-tools/heart-cell-atlas-scvi',
    cache_dir='/content/hf_cache',
)
print('Downloaded.')
print(hub_model.model_card)

In [ ]:
# The pretrained model expects a specific gene panel from ITS training atlas.
# load_query_data() aligns our genes to that panel (filling in missing genes
# as zero) rather than retraining anything — this is the standard scvi-tools
# 'query' workflow for adapting a foreign pretrained model to new data.
import scvi

pretrained_model = hub_model.model
query_model = scvi.model.SCVI.load_query_data(
    adata,
    pretrained_model,
)
query_model.train(max_epochs=20, plan_kwargs={'weight_decay': 0.0})  # light fine-tune only
print('Query adaptation complete.')

In [ ]:
# Extract latent representation from the FOREIGN-pretrained, query-adapted model
latent_hf = query_model.get_latent_representation()
print(f'Latent shape: {latent_hf.shape}')
adata.obsm['X_scVI_hf'] = latent_hf

In [ ]:
# Same honest evaluation as the from-scratch notebook — patient-level CV,
# same classifiers, same hyperparameters. Only the input features differ.
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
import xgboost as xgb

X = latent_hf
y = (adata.obs['disease'] == DCM).astype(int).values
groups = adata.obs['donor_id'].astype('category').cat.codes.values
scale = y.sum() / (1 - y).sum()

results_hf = {}
for name, clf in [
    ('RF (HF pretrained)',  RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)),
    ('XGB (HF pretrained)', xgb.XGBClassifier(n_estimators=200, scale_pos_weight=scale, random_state=42, eval_metric='logloss', verbosity=0))
]:
    aucs = []
    for fold, (tr, val) in enumerate(StratifiedGroupKFold(n_splits=5).split(X, y, groups)):
        clf.fit(X[tr], y[tr])
        auc = roc_auc_score(y[val], clf.predict_proba(X[val])[:,1])
        aucs.append(auc)
        print(f'{name} fold {fold+1}: {auc:.4f}')
    results_hf[name] = aucs
    print(f'  Mean: {np.mean(aucs):.4f}\n')

print('=== THREE-WAY COMPARISON (fill in scvi_pretraining.ipynb numbers once that run finishes) ===')
print('Raw genes        RF:  0.705   XGB: 0.701')
print('Own scVI latent  RF:  [from scvi_pretraining.ipynb]   XGB: [from scvi_pretraining.ipynb]')
print(f'HF pretrained    RF:  {np.mean(results_hf["RF (HF pretrained)"]):.4f}   XGB: {np.mean(results_hf["XGB (HF pretrained)"]):.4f}')

## Reading the three-way result

**If HF-pretrained AUC is close to raw-gene AUC (~0.70):** the foreign representation isn't adding much — plausible, since it was never optimized for this cohort's genes or patients.

**If HF-pretrained AUC is close to or better than own-scVI AUC:** transfer learning from a large public atlas is competitive with training your own model — a genuinely interesting finding, since it means you don't need to solve the RAM/training-time problem to get a useful representation.

**If HF-pretrained AUC is noticeably worse than both:** the query-adaptation didn't transfer well, likely due to gene panel mismatch between the two atlases — expected, and worth naming rather than hiding.

Whatever happens, this result answers 'does a public pretrained checkpoint transfer to our task,' not 'is pretraining worth doing.' Only the from-scratch run in `scvi_pretraining.ipynb` answers the second question.